# DS 862 Final Project — Credit Risk 



## 1. Business problem 

- **Outcome:** `SeriousDlqin2yrs` — serious delinquency (90+ days) within two years.
- **Use case:** support credit underwriting / limit decisions (edit to match your narrative).
- **Decision:** choose a probability threshold (and optional budget) aligned with costs/benefits in Section 10.

*Edit the cell above or duplicate into your report’s introduction.*

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Dataset path: Project_DS_862/cs-training.csv/cs-training.csv
DATA_PATH = Path("cs-training.csv") / "CRA.csv"
assert DATA_PATH.resolve().exists(), f"Data not found at {DATA_PATH.resolve()} — cwd is {Path.cwd()}"

## 2. Load data & variable inventory 

In [ ]:
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
display(df.head())
display(df.dtypes.to_frame("dtype"))
TARGET = "SeriousDlqin2yrs"
ID_COL = "Unnamed: 0"
FEATURES = [c for c in df.columns if c not in (TARGET, ID_COL)]

## 3. Exploratory data analysis 



In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("n_missing"))

rate = df[TARGET].mean()
print(f"Positive rate ({TARGET}==1): {rate:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#4c72b0", "#c44e52"])
ax.set_xticklabels(["No serious DQ", "Serious DQ"], rotation=0)
ax.set_title("Class balance")
plt.tight_layout()
plt.show()

desc = df[FEATURES + [TARGET]].describe().T
display(desc)

In [ ]:
corr = df[FEATURES + [TARGET]].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=False)
plt.title("Correlation heatmap (raw data, includes missing as NaN in corr)")
plt.tight_layout()
plt.show()

## 4. Data cleaning & preprocessing 



In [ ]:
def clean_credit_frame(d: pd.DataFrame) -> pd.DataFrame:
    out = d.drop(columns=[ID_COL], errors="ignore").copy()
    # Invalid ages → NaN → imputer handles
    out.loc[out["age"] <= 0, "age"] = np.nan
    # Competition data often has absurd utilization / debt ratio tails
    util_cap = 1.5
    out["RevolvingUtilizationOfUnsecuredLines"] = out["RevolvingUtilizationOfUnsecuredLines"].clip(
        upper=util_cap
    )
    debt_cap = 10.0
    out["DebtRatio"] = out["DebtRatio"].clip(upper=debt_cap)
    return out


df_clean = clean_credit_frame(df)
y = df_clean[TARGET].astype(int)
X = df_clean[FEATURES]

## 5. Train / test split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(X_train.shape, X_test.shape)

## 6. Baseline & candidate models 



In [ ]:
numeric_features = FEATURES

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        )
    ]
)

log_reg = Pipeline(
    steps=[
        ("prep", preprocess),
        (
            "clf",
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        ),
    ]
)

rf_clf = Pipeline(
    steps=[
        ("prep", preprocess),
        (
            "clf",
            RandomForestClassifier(
                random_state=RANDOM_STATE,
                class_weight="balanced_subsample",
                n_jobs=-1,
            ),
        ),
    ]
)

In [ ]:
def eval_classifier(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        "model": name,
        "roc_auc": roc_auc_score(y_te, proba),
        "pr_auc": average_precision_score(y_te, proba),
    }
    print(name, "ROC-AUC:", round(out["roc_auc"], 4), "PR-AUC:", round(out["pr_auc"], 4))
    print(classification_report(y_te, pred, digits=4))
    print("confusion_matrix @0.5:\n", confusion_matrix(y_te, pred))
    return model, proba, out


results = []
trained = {}
for name, m in [("LogisticRegression", log_reg), ("RandomForest (initial)", rf_clf)]:
    model, proba, meta = eval_classifier(name, m, X_train, y_train, X_test, y_test)
    trained[name] = (model, proba)
    results.append(meta)

pd.DataFrame(results)

## 7. Cross-validation & hyperparameter tuning 



In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grid_rf = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth": [None, 8],
    "clf__min_samples_leaf": [1, 5],
}

grid_rf = GridSearchCV(
    estimator=rf_clf,
    param_grid=param_grid_rf,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
)
grid_rf.fit(X_train, y_train)
print("Best ROC-AUC (CV):", grid_rf.best_score_)
print("Best params:", grid_rf.best_params_)

best_rf = grid_rf.best_estimator_
_, proba_rf_best, _ = eval_classifier("RandomForest (tuned)", best_rf, X_train, y_train, X_test, y_test)

## 8. ROC & Precision–Recall curves 



In [ ]:
primary_model = best_rf
primary_name = "RandomForest (tuned)"

y_score = primary_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_score)
prec, rec, _ = precision_recall_curve(y_test, y_score)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_score):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR")
axes[0].set_title(f"ROC — {primary_name}")
axes[0].legend()

axes[1].plot(rec, prec, label=f"AP = {average_precision_score(y_test, y_score):.3f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title(f"PR — {primary_name}")
axes[1].legend()
plt.tight_layout()
plt.show()

## 9. Threshold tuning & cost / profit view



In [ ]:
def expected_value_per_record(y_true, y_proba, threshold, payoff_fn):
    """payoff_fn(y_true, y_pred) -> per-row payoff."""
    y_pred = (y_proba >= threshold).astype(int)
    pays = np.array([payoff_fn(int(yt), int(yp)) for yt, yp in zip(y_true, y_pred)])
    return pays.mean()


# Illustrative constants — REPLACE with values justified in your report
B_TP = 1.0   # benefit of catching a bad (e.g. loss avoided, normalized)
C_FP = 0.3   # cost of wrongly flagging a good customer
C_FN = 1.2   # cost of missing a bad


def credit_payoff(y_true, y_pred):
    if y_pred == 1 and y_true == 1:
        return B_TP
    if y_pred == 1 and y_true == 0:
        return -C_FP
    if y_pred == 0 and y_true == 1:
        return -C_FN
    return 0.0


thresholds = np.linspace(0.02, 0.6, 200)
evs = [expected_value_per_record(y_test.values, y_score, t, credit_payoff) for t in thresholds]
best_t = thresholds[int(np.argmax(evs))]
print("Best threshold (illustrative payoffs):", round(best_t, 4))
print("Max EV per record:", round(max(evs), 5))

plt.figure(figsize=(7, 4))
plt.plot(thresholds, evs)
plt.axvline(best_t, color="C1", ls="--", label=f"best t={best_t:.3f}")
plt.xlabel("Threshold")
plt.ylabel("Expected value / record")
plt.title("Threshold sweep (edit payoffs in code)")
plt.legend()
plt.tight_layout()
plt.show()

y_hat_star = (y_score >= best_t).astype(int)
print("Confusion matrix @ business threshold:\n", confusion_matrix(y_test, y_hat_star))
print(classification_report(y_test, y_hat_star, digits=4))

## 10.  PCA (2D) for visualization


In [ ]:
from sklearn.decomposition import PCA
Xt = preprocess.fit_transform(X_train)
Z = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xt)
plt.figure(figsize=(6,5))
plt.scatter(Z[:,0], Z[:,1], c=y_train, s=6, alpha=0.35, cmap="coolwarm")
plt.colorbar(label=TARGET)
plt.title("PCA of training features (scaled, imputed)")
plt.tight_layout()
plt.show()